# MH-01. MH PheCode nodes + cohort + spine skeleton

Broad mental-disorders cohort (NOT psychosis-risk). Locked decisions:

- Cohort = persons with at least one condition mapping to an MH-category PheCode (v1.2),
  AND whose earliest MH dx (index) falls at age 18-35 inclusive. Earlier-than-18 index
  is excluded (index is NOT reset to a later dx).
- index_date = min(condition_start_date) over MH dx. Tie-break: smallest 3-digit parent,
  then smallest full PheCode (numeric).
- ICD-9-CM / ICD-10-CM -> PheCode only. SNOMED-only dx are counted as unmapped (not mapped).
- years_in_ehr_before_index derives from the first/last raw EHR event across the full tables\n  (this release has no observation_period table); single source, reused in MH-04.

Scope cuts (agreed): no FAMD/MFA, no ADI/SVI, no note_nlp, no SNOMED mapping.

Output: `output/mh/spine.parquet` + node list + unmapped counts.


## 1. Config and external file checks

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb

DATA_PATH = '/home/jupyter/2836994-data2'      # same OMOP release as psychosis pipeline
PHECODE_MAP_PATH = 'phecode_icd10.csv'         # ICD -> PheCode crosswalk (confirm ICD-9 + ICD-10)
PHECODE_DEF_PATH = 'phecode_definitions1.2.csv'  # PheCode -> description + category (confirm exists)
OUTPUT_DIR = Path('output/mh')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Locked cohort parameters
AGE_MIN, AGE_MAX = 18, 35          # inclusive; earlier-than-min index is excluded, not reset
CROSSWALK_VERSION = 'phecode_v1.2' # recorded in provenance

# MH 3-digit parent categories (v1.2 mental disorders)
MH_PARENTS = {295, 296, 300, 301, 303, 304, 305, 306, 313, 316, 317, 318}
PSYCHOSIS_PARENT = 295             # 295.x force-retained downstream; later-psychosis outcome

con = duckdb.connect()
print('DuckDB:', duckdb.__version__)
for p in [PHECODE_MAP_PATH, PHECODE_DEF_PATH]:
    print(f'{p:40s} exists = {Path(p).exists()}')
print('DATA_PATH:', DATA_PATH)


## 2. Build MH PheCode node set from the crosswalk

In [ ]:
# ICD -> PheCode map. Expect columns like: icd, phecode (+ maybe flag for ICD-9/10).
phe_map = pd.read_csv(PHECODE_MAP_PATH, dtype=str)
phe_map.columns = [c.strip().lower() for c in phe_map.columns]
print('crosswalk columns:', list(phe_map.columns))

ICD_COL = 'icd' if 'icd' in phe_map.columns else phe_map.columns[0]
PHE_COL = 'phecode' if 'phecode' in phe_map.columns else phe_map.columns[1]
phe_map = phe_map[[ICD_COL, PHE_COL]].dropna().rename(columns={ICD_COL:'icd', PHE_COL:'phecode'})
phe_map['icd'] = phe_map['icd'].str.strip().str.upper()
phe_map['phecode'] = phe_map['phecode'].str.strip()

def parent3(phecode):
    try:
        return int(float(phecode))     # 3-digit integer parent, e.g. 296.2 -> 296
    except Exception:
        return np.nan

phe_map['parent3'] = phe_map['phecode'].map(parent3)
mh_map = phe_map[phe_map['parent3'].isin(MH_PARENTS)].copy()
mh_nodes = sorted(mh_map['phecode'].unique(), key=lambda s: float(s))
print(f'MH ICD->PheCode rows: {len(mh_map):,} | distinct MH nodes: {len(mh_nodes)}')

# PheCode labels + category (for index_concept_label and MH category names)
if Path(PHECODE_DEF_PATH).exists():
    defs = pd.read_csv(PHECODE_DEF_PATH, dtype=str)
    defs.columns = [c.strip().lower() for c in defs.columns]
    label_col = next((c for c in ['phenotype','description','phecode_str'] if c in defs.columns), None)
    node_labels = (defs[['phecode', label_col]].rename(columns={label_col:'label'})
                   if label_col else pd.DataFrame(columns=['phecode','label']))
else:
    print('WARNING: PHECODE_DEF_PATH missing -> labels will be blank, category from parent3 only')
    node_labels = pd.DataFrame({'phecode': mh_nodes, 'label': ['']*len(mh_nodes)})

node_table = (pd.DataFrame({'phecode': mh_nodes})
              .assign(parent3=lambda d: d['phecode'].map(parent3))
              .merge(node_labels, on='phecode', how='left'))
node_table['is_psychosis'] = node_table['parent3'] == PSYCHOSIS_PARENT
node_table.to_parquet(OUTPUT_DIR / 'mh_node_table.parquet', index=False)
print('Saved node table:', node_table.shape)
node_table.head()


## 3. Map each patient's diagnoses to MH PheCodes (ICD-9/10 only)

In [ ]:
# Route via OMOP concept: condition_source_concept_id -> concept.concept_code (the raw
# source-vocabulary code, ICD9CM or ICD10CM) -> normalize -> join the phecode crosswalk.
# Also keep a direct condition_source_value join as a UNION fallback (some rows may have
# source_concept_id = 0 but a usable source_value). Works whichever vocab the crosswalk covers.
#
# PREREQUISITE: concept/*.parquet must exist under DATA_PATH. If not, this path is unavailable.
import glob
assert glob.glob(f'{DATA_PATH}/concept/*.parquet'), \
    f'No concept table under {DATA_PATH}/concept -> cannot use the OMOP concept route.'

def to_dotted(s):
    """Uppercase/trim; insert a dot after the 3rd char for undotted codes > 3 chars
    (F209 -> F20.9, 29510 -> 295.10). Dotted or short codes unchanged."""
    s = s.astype('string').str.upper().str.strip()
    no_dot = ~s.str.contains(r'\\.', na=False) & (s.str.len() > 3)
    return s.mask(no_dot, s.str.slice(0, 3) + '.' + s.str.slice(3))

mh_map = mh_map.copy()
mh_map['icd_dot'] = to_dotted(mh_map['icd'])
con.register('mh_map', mh_map[['icd', 'icd_dot', 'phecode', 'parent3']])
phe_all = phe_map[['icd']].copy(); phe_all['icd_dot'] = to_dotted(phe_all['icd'])
con.register('phe_all', phe_all[['icd_dot']].drop_duplicates())

def gp(table):
    return f"read_parquet('{DATA_PATH}/{table}/*.parquet')"

# Normalized source codes per condition row, from BOTH concept_code and source_value.
# concept_code / condition_source_value get dots inserted in SQL via a small helper expression.
DOTIFY = (lambda col:
    f"CASE WHEN POSITION('.' IN {col}) > 0 OR LENGTH({col}) <= 3 THEN UPPER(TRIM({col})) "
    f"ELSE UPPER(TRIM(SUBSTR({col},1,3))) || '.' || UPPER(TRIM(SUBSTR({col},4))) END")

src_codes_sql = f'''
WITH rows AS (
  SELECT c.person_id,
         COALESCE(c.condition_start_date, CAST(c.condition_start_datetime AS DATE)) AS dx_date,
         co.concept_code   AS concept_code,
         co.vocabulary_id  AS vocab,
         c.condition_source_value AS src_val
  FROM {gp('condition_occurrence')} c
  LEFT JOIN {gp('concept')} co
    ON c.condition_source_concept_id = co.concept_id
)
SELECT person_id, dx_date, vocab,
       {DOTIFY('concept_code')} AS code_from_concept,
       {DOTIFY('src_val')}      AS code_from_srcval
FROM rows
WHERE dx_date IS NOT NULL
'''
con.sql(f'CREATE OR REPLACE TEMP VIEW src_codes AS {src_codes_sql}')

# --- Diagnostic: what vocabularies do we actually have, and how many hit the crosswalk? ---
vocab_hits = con.sql('''
SELECT s.vocab,
       COUNT(*) AS n_rows,
       COUNT(*) FILTER (WHERE m.icd_dot IS NOT NULL) AS n_hit_concept,
       COUNT(*) FILTER (WHERE m2.icd_dot IS NOT NULL) AS n_hit_srcval
FROM src_codes s
LEFT JOIN mh_map m  ON s.code_from_concept = m.icd_dot
LEFT JOIN mh_map m2 ON s.code_from_srcval  = m2.icd_dot
GROUP BY s.vocab ORDER BY n_rows DESC LIMIT 15
''').to_df()
print('rows + MH-crosswalk hits by source vocabulary:')
print(vocab_hits)

# --- MH mapping: match on concept_code OR source_value, dedup to one (person,date,phecode) ---
mh_dx = con.sql('''
SELECT DISTINCT s.person_id, s.dx_date, m.phecode, m.parent3
FROM src_codes s
JOIN mh_map m
  ON m.icd_dot = s.code_from_concept OR m.icd_dot = s.code_from_srcval
''').to_df()
mh_dx['dx_date'] = pd.to_datetime(mh_dx['dx_date'])
print('MH-mapped dx rows:', f'{len(mh_dx):,}', '| persons:', f"{mh_dx['person_id'].nunique():,}")

if len(mh_dx) == 0:
    print('STILL 0 -> check vocab_hits above: if you see ICD10CM rows but 0 hits, the crosswalk '
          'is ICD-9 only (need an ICD-10-CM phecode map). If concept_code is mostly SNOMED, the '
          'source_concept_id points at standard concepts, not source ICD.')

# --- Unmapped: rows whose codes match NO phecode in the full crosswalk ---
unmapped = con.sql('''
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT person_id) AS n_persons
FROM src_codes s
LEFT JOIN phe_all a ON s.code_from_concept = a.icd_dot
LEFT JOIN phe_all b ON s.code_from_srcval  = b.icd_dot
WHERE a.icd_dot IS NULL AND b.icd_dot IS NULL
''').to_df()
print('Rows not mapping to ANY phecode (SNOMED-only + out-of-vocab):')
print(unmapped)


## 4. Index date, tie-break, and 18-35 age gate

In [ ]:
# Attach birth_datetime for age_at_index.
persons = con.sql(f"SELECT person_id, birth_datetime, gender_source_value FROM {gp('person')}").to_df()
persons['birth_datetime'] = pd.to_datetime(persons['birth_datetime'])

# Earliest MH dx date per person = candidate index.
idx_date = mh_dx.groupby('person_id', as_index=False)['dx_date'].min().rename(columns={'dx_date':'index_date'})

# Tie-break among dx on the index date: smallest parent3, then smallest full phecode.
tie = mh_dx.merge(idx_date, on='person_id')
tie = tie[tie['dx_date'] == tie['index_date']].copy()
tie['phe_num'] = tie['phecode'].astype(float)
tie = tie.sort_values(['person_id','parent3','phe_num'])
index_pick = tie.groupby('person_id', as_index=False).first()[['person_id','index_date','parent3','phecode']]
index_pick = index_pick.rename(columns={'parent3':'index_signal_type','phecode':'index_phecode'})

spine = index_pick.merge(persons, on='person_id', how='left')
spine['age_at_index'] = ((spine['index_date'] - spine['birth_datetime']).dt.days // 365.25).astype('Int64')

before = len(spine)
spine = spine[(spine['age_at_index'] >= AGE_MIN) & (spine['age_at_index'] <= AGE_MAX)].copy()
print(f'Persons with MH dx: {before:,} -> after 18-35 index gate: {len(spine):,}')

# Assertions: gate is exactly inclusive and index is the earliest MH dx.
assert spine['age_at_index'].between(AGE_MIN, AGE_MAX).all()
assert (spine.merge(idx_date, on='person_id')['index_date_x']
        .eq(spine.merge(idx_date, on='person_id')['index_date_y']).all())
print('age gate + index assertions passed')


## 5. Spine fields from raw EHR event span (no observation_period table)

In [ ]:
# years_in_ehr_before_index + post_index_observation_days.
# This OMOP release has NO observation_period table, so derive the EHR span directly
# from the raw full event tables: op_start = first ever event, op_end = last ever event.
# This single computed field is the SOLE lookback filter in MH-04.
import glob

EHR_SPAN_TABLES = [
    ('condition_occurrence', 'condition_start_date'),
    ('visit_occurrence',     'visit_start_date'),
    ('measurement',          'measurement_date'),
    ('drug_exposure',        'drug_exposure_start_date'),
    ('procedure_occurrence', 'procedure_date'),
    ('observation',          'observation_date'),
]
present = [(t, c) for (t, c) in EHR_SPAN_TABLES if glob.glob(f'{DATA_PATH}/{t}/*.parquet')]
print('EHR-span tables present:', [t for t, _ in present])
if not present:
    raise FileNotFoundError('No event tables found under DATA_PATH to derive EHR span.')

union_sql = ' UNION ALL '.join(
    f'SELECT person_id, {c} AS d FROM {gp(t)} WHERE {c} IS NOT NULL' for t, c in present
)
op = con.sql(f'''
SELECT person_id, MIN(d) AS op_start, MAX(d) AS op_end
FROM ({union_sql})
GROUP BY person_id
''').to_df()
op['op_start'] = pd.to_datetime(op['op_start'])
op['op_end'] = pd.to_datetime(op['op_end'])

spine = spine.merge(op, on='person_id', how='left')
spine['years_in_ehr_before_index'] = (spine['index_date'] - spine['op_start']).dt.days / 365.25
spine['post_index_observation_days'] = (spine['op_end'] - spine['index_date']).dt.days
spine['years_in_ehr_before_index'] = spine['years_in_ehr_before_index'].clip(lower=0)
spine['post_index_observation_days'] = spine['post_index_observation_days'].clip(lower=0)

# index_concept_label from the node table.
spine = spine.merge(node_table[['phecode','label']].rename(columns={'phecode':'index_phecode',
                    'label':'index_concept_label'}), on='index_phecode', how='left')
spine['sex'] = spine['gender_source_value']

# all_negative / all_masked are filled in MH-02 (need Y); placeholders here.
spine['all_negative'] = pd.NA
spine['all_masked'] = pd.NA

spine = spine.sort_values('person_id').reset_index(drop=True)
spine_cols = ['person_id','index_date','age_at_index','index_signal_type','index_phecode',
              'index_concept_label','sex','years_in_ehr_before_index',
              'post_index_observation_days','all_negative','all_masked']
spine = spine[spine_cols]
spine.to_parquet(OUTPUT_DIR / 'spine.parquet', index=False)
mh_dx.sort_values('person_id').to_parquet(OUTPUT_DIR / 'mh_dx_events.parquet', index=False)

print('spine:', spine.shape)
print('index_signal_type distribution:')
print(spine['index_signal_type'].value_counts().sort_index())
spine.head()


## 6. Provenance stub

In [ ]:
prov = {
    'crosswalk_version': CROSSWALK_VERSION,
    'age_gate': [AGE_MIN, AGE_MAX],
    'n_mh_nodes': len(mh_nodes),
    'n_cohort': int(len(spine)),
    'unmapped_rows': int(unmapped['n_rows'].iloc[0]),
    'unmapped_persons': int(unmapped['n_persons'].iloc[0]),
    'scope_cuts': ['no_FAMD','no_ADI_SVI','no_note_nlp','no_SNOMED_mapping'],
}
import json
(OUTPUT_DIR / 'provenance_mh01.json').write_text(json.dumps(prov, indent=2))
print(json.dumps(prov, indent=2))
